# Stable Diffusion XL (SDXL) Porting to LiteRT (TFLite)

이 노트북은 Stability AI의 SDXL Base 모델을 LiteRT Edge Generative API를 사용하여
TFLite 모델로 변환하고, 모바일 디바이스에서 실행 가능한 형태로 포팅하는 전체 과정을 보여줍니다.

## 전체 워크플로우

1. **환경 설정** - 필요한 패키지 설치
2. **체크포인트 다운로드** - HuggingFace에서 SDXL 가중치 다운로드
3. **SDXL 아키텍처 이해** - SD 1.5와의 차이점
4. **모델 재작성 (Re-author)** - Edge Generative API 빌딩 블록으로 모델 구성
5. **설정 검증** - 각 컴포넌트의 설정 및 파라미터 수 확인
6. **가중치 로드** - 원본 체크포인트에서 재작성 모델로 가중치 매핑
7. **변환 (Convert)** - PyTorch → TFLite 변환
8. **양자화 (Quantize)** - 모델 크기 및 성능 최적화
9. **추론 (Inference)** - 변환된 TFLite 모델로 이미지 생성
10. **Playground** - 다양한 프롬프트와 설정으로 실험

## 1. 환경 설정

In [ ]:
# 필수 패키지 설치
!pip install -e ./litert-torch
!pip install safetensors pillow tqdm numpy regex
!pip install huggingface-hub[xet] transformers tensorflow
!pip install --upgrade ai_edge_litert ai_edge_quantizer

In [ ]:
import os
import pathlib

import litert_torch
import litert_torch.generative.layers.builder as layers_builder
import litert_torch.generative.layers.model_config as layers_cfg
from litert_torch.generative.layers.unet import blocks_2d
import litert_torch.generative.layers.unet.model_config as unet_cfg
from litert_torch.generative.layers.attention import TransformerBlock
import litert_torch.generative.layers.attention_utils as attention_utils
from litert_torch.generative.quantize import quant_recipes
from litert_torch.generative.utilities import stable_diffusion_loader
import litert_torch.generative.utilities.loader as loading_utils
import numpy as np
from PIL import Image
import torch
from torch import nn
import tqdm

print(f"LiteRT Torch version: {litert_torch.__version__}")
print(f"PyTorch version: {torch.__version__}")

## 2. 체크포인트 다운로드

HuggingFace에서 SDXL Base 1.0의 safetensors 체크포인트를 다운로드합니다.

사전에 `huggingface-cli`로 로그인하거나, 수동으로 다운로드하여 경로를 설정하세요.

In [ ]:
# 체크포인트 경로 설정
# HuggingFace에서 stabilityai/stable-diffusion-xl-base-1.0 다운로드
#
# 방법 1: huggingface-cli 사용
# !huggingface-cli download stabilityai/stable-diffusion-xl-base-1.0 \
#     sd_xl_base_1.0.safetensors \
#     --local-dir ~/stable-diffusion-xl/
#
# 방법 2: git lfs 사용
# !git lfs install
# !git clone https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0 ~/stable-diffusion-xl

CKPT_DIR = os.path.join(pathlib.Path.home(), "stable-diffusion-xl")
CKPT_PATH = os.path.join(CKPT_DIR, "sd_xl_base_1.0.safetensors")
TOKENIZER_DIR = os.path.join(CKPT_DIR, "tokenizer")
OUTPUT_DIR = "/tmp/sdxl_tflite"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Checkpoint path: {CKPT_PATH}")
print(f"Checkpoint exists: {os.path.exists(CKPT_PATH)}")
print(f"Tokenizer dir: {TOKENIZER_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

## 3. SDXL 아키텍처 이해

SDXL은 SD 1.5 대비 상당한 아키텍처 차이가 있습니다:

| 항목 | SD 1.5 | SDXL |
|------|--------|------|
| **텍스트 인코더** | CLIP (768-dim, 12L) | CLIP-L (768-dim, 12L) + OpenCLIP-G (1280-dim, 32L) |
| **Context Dim** | 768 | 2048 (768+1280 concat) |
| **UNet 스테이지** | 4 [320,640,1280,1280] | 3 [320,640,1280] |
| **Transformer 레이어/블록** | 1 | 0, 2, 또는 10 (블록마다 다름) |
| **Attention Heads** | 8 (고정) | head_dim=64 → 5/10/20 (블록마다 다름) |
| **추가 조건** | 없음 | add_embedding (pooled text + time_ids) |
| **VAE Scaling Factor** | 0.18215 | 0.13025 |
| **이미지 크기** | 512×512 | 1024×1024 |
| **체크포인트 Prefix** | `cond_stage_model.*` | `conditioner.embedders.*` |

### SDXL 핵심 컴포넌트

| 컴포넌트 | 역할 | 입력 | 출력 |
|---------|------|------|------|
| **CLIP-L** | 텍스트 → 768-dim 임베딩 | 토큰 (77,) | hidden_states (1, 77, 768) |
| **OpenCLIP-G** | 텍스트 → 1280-dim 임베딩 + pooled | 토큰 (77,) | hidden (1, 77, 1280) + pooled (1, 1280) |
| **SDXL UNet** | 노이즈 제거 | 잠재 텐서 + context + time + add_emb | 노이즈 예측 |
| **VAE Decoder** | 잠재 공간 → 이미지 | 잠재 (1, 4, 128, 128) | 이미지 (1, 3, 1024, 1024) |

### 듀얼 텍스트 인코더 아키텍처

```
                  tokens (77)
                     |
           ┌─────────┴─────────┐
           ▼                   ▼
       CLIP-L (12L)      OpenCLIP-G (32L)
     768-dim hidden     1280-dim penultimate
           |            + 1280-dim pooled
           └──────┬──────┘       |
                  ▼              ▼
        context (77, 2048)  add_embedding
                  |         (2816 → 1280)
                  └────┬────┘
                       ▼
                   SDXL UNet
```

## 4. 모델 재작성 (Re-authoring)

Edge Generative API의 빌딩 블록을 사용하여 SDXL 컴포넌트를 재작성합니다.

### 사용하는 빌딩 블록:
- `TransformerBlock` - 텍스트 인코더 트랜스포머 블록
- `blocks_2d.MultiLayerTransformerBlock2D` - SDXL UNet의 다중 레이어 트랜스포머 (NEW)
- `blocks_2d.DownEncoderBlock2D` - UNet 다운 인코더
- `blocks_2d.MidBlock2D` - UNet 미드 블록
- `blocks_2d.SkipUpDecoderBlock2D` - UNet 업 디코더 (스킵 커넥션)
- `blocks_2d.UpDecoderBlock2D` - VAE 디코더의 업 디코더

### 4.1 CLIP-L Text Encoder

CLIP-L은 SD 1.5와 동일한 구조입니다.
차이점: SDXL에서는 final_layer_norm 적용 전의 hidden states를 사용합니다.
체크포인트 prefix: `conditioner.embedders.0.*`

In [ ]:
from litert_torch.generative.examples.stable_diffusion_xl import clip

print("=== CLIP-L Tensor Names ===")
print(f"  Q proj: {clip.TENSOR_NAMES.attn_query_proj}")
print(f"  Embedding: {clip.TENSOR_NAMES.embedding}")
print()
print("CLIP-L returns hidden states BEFORE final norm (for SDXL context).")

### 4.2 OpenCLIP-G Text Encoder

OpenCLIP-G (ViT-bigG)는 SDXL의 두 번째 텍스트 인코더입니다.

주요 특징:
- 32 layers, 1280 embedding dim, 20 attention heads
- GELU 활성화 (CLIP-L의 GELU_QUICK과 다름)
- Fused QKV (in_proj_weight) 형식의 체크포인트
- Penultimate hidden states (마지막에서 두 번째 레이어 출력) 반환
- Pooled output: EOS 토큰 → text_projection
- 체크포인트 prefix: `conditioner.embedders.1.model.*`

In [ ]:
from litert_torch.generative.examples.stable_diffusion_xl import open_clip

open_clip_config = open_clip.get_model_config()
print("=== OpenCLIP-G Config ===")
print(f"  vocab_size: {open_clip_config.vocab_size}")
print(f"  num_layers: {open_clip_config.num_layers}")
print(f"  embedding_dim: {open_clip_config.embedding_dim}")
print(f"  max_seq_len: {open_clip_config.max_seq_len}")
print(f"  num_heads: {open_clip_config.block_config(0).attn_config.num_heads}")
print(f"  head_dim: {open_clip_config.block_config(0).attn_config.head_dim}")

### 4.3 SDXL UNet (Diffusion Model)

SDXL UNet은 SD 1.5 대비 가장 큰 변경점입니다.

```
latents ─► ConvIn ─► DownEncoder(x3) ─► MidBlock ─► SkipUpDecoder(x3) ─► Norm ─► Act ─► ConvOut
                         │                                    ▲
                         └──── skip connections ──────────────┘
```

**블록별 설정:**

| 블록 | 채널 | Transformer 레이어 | Attention Heads | 다운/업 샘플링 |
|------|------|-------------------|----------------|---------------|
| Down 0 | 320 | 0 (없음) | - | ✓ |
| Down 1 | 640 | 2 | 10 | ✓ |
| Down 2 | 1280 | 10 | 20 | ✗ |
| Mid | 1280 | 10 | 20 | - |
| Up 0 | 1280 | 10 | 20 | ✓ |
| Up 1 | 640 | 2 | 10 | ✓ |
| Up 2 | 320 | 0 (없음) | - | ✗ |

**AddEmbedding** (SDXL 전용):
- 입력: pooled_text_embed(1280) + time_ids_embed(1536) = 2816
- 구조: Linear(2816→1280) + SiLU + Linear(1280→1280)
- time_emb에 더해져서 UNet에 전달됨

In [ ]:
from litert_torch.generative.examples.stable_diffusion_xl import diffusion

print("=== SDXL UNet Architecture ===")
print(f"  Block out channels: {diffusion._BLOCK_OUT_CHANNELS}")
print(f"  Transformer layers per block: {diffusion._TRANSFORMER_LAYERS_PER_BLOCK}")
print(f"  Head dim: {diffusion._HEAD_DIM}")
print(f"  Cross attention dim: {diffusion._CROSS_ATTENTION_DIM}")
print(f"  Time embedding dim: {diffusion._TIME_EMBEDDING_DIM}")
print(f"  Add embedding in dim: {diffusion._ADD_EMBEDDING_IN_DIM}")
print(f"  Layers per block: {diffusion._LAYERS_PER_BLOCK}")
print(f"  Mid block transformer layers: {diffusion._MID_BLOCK_TRANSFORMER_LAYERS}")

### 4.4 VAE Decoder

VAE Decoder는 SD 1.5와 동일한 구조이며, `scaling_factor=0.13025`만 다릅니다.

In [ ]:
from litert_torch.generative.examples.stable_diffusion_xl import decoder

dec_config = decoder.get_model_config()
print("=== SDXL VAE Decoder Config ===")
print(f"  latent_channels: {dec_config.latent_channels}")
print(f"  out_channels: {dec_config.out_channels}")
print(f"  block_out_channels: {dec_config.block_out_channels}")
print(f"  scaling_factor: {dec_config.scaling_factor} (SD 1.5: 0.18215)")
print(f"  layers_per_block: {dec_config.layers_per_block}")

## 5. 설정 검증 및 파라미터 수 확인

In [ ]:
from litert_torch.generative.examples.stable_diffusion_xl import util

DEVICE_TYPE = "gpu"  # GPU 백엔드 최적화 (StableHLO composite ops 활성화)

# CLIP-L
clip_config = clip.get_model_config()
clip_model = clip.CLIPEncoder(clip_config)
clip_params = sum(p.numel() for p in clip_model.parameters())
print(f"CLIP-L parameters: {clip_params:,}")

# OpenCLIP-G
open_clip_model = open_clip.OpenCLIP(open_clip.get_model_config())
open_clip_params = sum(p.numel() for p in open_clip_model.parameters())
print(f"OpenCLIP-G parameters: {open_clip_params:,}")

# SDXL UNet (GPU 최적화)
unet_model = diffusion.SDXLDiffusion(batch_size=2, device_type=DEVICE_TYPE)
unet_params = sum(p.numel() for p in unet_model.parameters())
print(f"SDXL UNet parameters: {unet_params:,} (device_type={DEVICE_TYPE})")

# VAE Decoder (GPU 최적화)
decoder_model = decoder.Decoder(decoder.get_model_config(device_type=DEVICE_TYPE))
decoder_params = sum(p.numel() for p in decoder_model.parameters())
print(f"VAE Decoder parameters: {decoder_params:,} (device_type={DEVICE_TYPE})")

total = clip_params + open_clip_params + unet_params + decoder_params
print(f"\nTotal parameters: {total:,} ({total / 1e9:.2f}B)")

## 6. 가중치 로드 및 모델 구성

원본 safetensors 체크포인트에서 가중치를 로드합니다.

In [ ]:
@torch.inference_mode()
def build_clip_model(ckpt_path: str) -> nn.Module:
    """CLIP-L 모델을 생성하고 가중치를 로드합니다."""
    model = clip.CLIPEncoder(clip.get_model_config())
    loader = stable_diffusion_loader.ClipModelLoader(
        ckpt_path, clip.TENSOR_NAMES
    )
    loader.load(model, strict=False)
    model.eval()
    print(f"CLIP-L loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model


@torch.inference_mode()
def build_open_clip_model(ckpt_path: str) -> nn.Module:
    """OpenCLIP-G 모델을 생성하고 가중치를 로드합니다."""
    model = open_clip.OpenCLIP(open_clip.get_model_config())
    loader = open_clip.OpenCLIPModelLoader(ckpt_path)
    loader.load(model, strict=False)
    model.eval()
    print(f"OpenCLIP-G loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model


@torch.inference_mode()
def build_diffusion_model(ckpt_path: str, batch_size: int = 2) -> nn.Module:
    """SDXL UNet 모델을 생성하고 가중치를 로드합니다. (GPU 최적화)"""
    model = diffusion.SDXLDiffusion(batch_size=batch_size, device_type=DEVICE_TYPE)
    loader = diffusion.SDXLDiffusionModelLoader(ckpt_path)
    loader.load(model, strict=False)
    model.eval()
    print(f"SDXL UNet loaded (device_type={DEVICE_TYPE}). Parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model


@torch.inference_mode()
def build_decoder_model(ckpt_path: str) -> nn.Module:
    """VAE Decoder 모델을 생성하고 가중치를 로드합니다. (GPU 최적화)"""
    model = decoder.Decoder(decoder.get_model_config(device_type=DEVICE_TYPE))
    loader = stable_diffusion_loader.AutoEncoderModelLoader(
        ckpt_path, decoder.TENSOR_NAMES
    )
    loader.load(model, strict=False)
    model.eval()
    print(f"VAE Decoder loaded (device_type={DEVICE_TYPE}). Parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model


print(f"Model builder functions defined. (device_type={DEVICE_TYPE})")

In [ ]:
# 모델 로드 (체크포인트가 있을 때만 실행)
if os.path.exists(CKPT_PATH):
    clip_loaded = build_clip_model(CKPT_PATH)
    open_clip_loaded = build_open_clip_model(CKPT_PATH)
    diffusion_loaded = build_diffusion_model(CKPT_PATH)
    decoder_loaded = build_decoder_model(CKPT_PATH)
    print("\nAll models loaded successfully!")
else:
    print(f"Checkpoint not found at: {CKPT_PATH}")
    print("Please download the checkpoint first (see Section 2).")

## 7. TFLite 변환 (Convert)

`litert_torch.signature()` API를 사용하여 4개 컴포넌트를 TFLite로 변환합니다.

In [ ]:
IMAGE_HEIGHT = 1024
IMAGE_WIDTH = 1024
N_TOKENS = 77


@torch.inference_mode()
def convert_all_to_tflite(
    clip_model,
    open_clip_model,
    diffusion_model,
    decoder_model,
    output_dir: str,
    quantize: bool = False,
):
    """모든 SDXL 컴포넌트를 TFLite로 변환합니다."""
    quant_config = quant_recipes.full_weight_only_recipe() if quantize else None
    if quantize:
        print("Quantization enabled: weight-only INT8")

    prompt_tokens = torch.full((1, N_TOKENS), 0, dtype=torch.int)

    # --- 1. CLIP-L ---
    print("\n[1/4] Converting CLIP-L text encoder...")
    litert_torch.signature(
        "encode", clip_model, (prompt_tokens,)
    ).convert(quant_config=quant_config).export(f"{output_dir}/clip.tflite")
    print(f"  Saved: {output_dir}/clip.tflite")

    # --- 2. OpenCLIP-G ---
    print("\n[2/4] Converting OpenCLIP-G text encoder...")
    litert_torch.signature(
        "encode", open_clip_model, (prompt_tokens,)
    ).convert(quant_config=quant_config).export(f"{output_dir}/open_clip.tflite")
    print(f"  Saved: {output_dir}/open_clip.tflite")

    # --- 3. SDXL UNet ---
    print("\n[3/4] Converting SDXL UNet...")
    input_latents = torch.zeros(
        (1, 4, IMAGE_HEIGHT // 8, IMAGE_WIDTH // 8), dtype=torch.float32
    )
    clip_out = clip_model(prompt_tokens)
    open_clip_out, pooled_out = open_clip_model(prompt_tokens)
    context_cond = torch.cat([clip_out, open_clip_out], dim=-1)
    context_uncond = torch.zeros_like(context_cond)
    context = torch.cat([context_cond, context_uncond], axis=0)

    time_embedding = util.get_time_embedding(0)
    add_time_ids = util.get_add_time_ids(
        original_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
        crop_coords=(0, 0),
        target_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    )
    time_ids_emb = util.encode_add_time_ids(add_time_ids)
    add_emb_cond = torch.cat([pooled_out, time_ids_emb], dim=-1)
    add_emb_uncond = torch.zeros_like(add_emb_cond)
    add_emb = torch.cat([add_emb_cond, add_emb_uncond], axis=0)

    litert_torch.signature(
        "diffusion",
        diffusion_model,
        (
            torch.repeat_interleave(input_latents, 2, 0),
            context,
            time_embedding,
            add_emb,
        ),
    ).convert(quant_config=quant_config).export(f"{output_dir}/diffusion.tflite")
    print(f"  Saved: {output_dir}/diffusion.tflite")

    # --- 4. VAE Decoder ---
    print("\n[4/4] Converting VAE Decoder...")
    litert_torch.signature(
        "decode", decoder_model, (input_latents,)
    ).convert(quant_config=quant_config).export(f"{output_dir}/decoder.tflite")
    print(f"  Saved: {output_dir}/decoder.tflite")

    print("\nConversion complete!")


print("Conversion function defined.")

In [ ]:
# 변환 실행 (양자화 없이)
if os.path.exists(CKPT_PATH):
    convert_all_to_tflite(
        clip_loaded,
        open_clip_loaded,
        diffusion_loaded,
        decoder_loaded,
        output_dir=OUTPUT_DIR,
        quantize=False,
    )
else:
    print("Skipping conversion - checkpoint not available.")

In [ ]:
# 변환된 TFLite 파일 크기 확인
for name in ["clip", "open_clip", "diffusion", "decoder"]:
    path = f"{OUTPUT_DIR}/{name}.tflite"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{name}.tflite: {size_mb:.1f} MB")
    else:
        print(f"{name}.tflite: not found")

## 8. 양자화 (Quantization)

모바일 배포를 위해 양자화를 적용합니다.

| 레시피 | 가중치 | 연산 | 특징 |
|--------|--------|------|------|
| `full_int8_dynamic_recipe()` | INT8 | INT | 최소 크기 |
| `full_weight_only_recipe()` | INT8 | FP | 안정적 품질 |
| `full_fp16_recipe()` | FP16 | FP | 최고 품질 |
| `full_int4_dynamic_block32_recipe()` | INT4 | INT | 극한 압축 |

In [ ]:
# 양자화 적용 변환 (weight-only INT8)
QUANTIZED_OUTPUT_DIR = "/tmp/sdxl_tflite_quantized"
os.makedirs(QUANTIZED_OUTPUT_DIR, exist_ok=True)

if os.path.exists(CKPT_PATH):
    convert_all_to_tflite(
        clip_loaded,
        open_clip_loaded,
        diffusion_loaded,
        decoder_loaded,
        output_dir=QUANTIZED_OUTPUT_DIR,
        quantize=True,
    )

    print("\n=== File Size Comparison ===")
    for name in ["clip", "open_clip", "diffusion", "decoder"]:
        orig = f"{OUTPUT_DIR}/{name}.tflite"
        quant = f"{QUANTIZED_OUTPUT_DIR}/{name}.tflite"
        if os.path.exists(orig) and os.path.exists(quant):
            orig_mb = os.path.getsize(orig) / (1024 * 1024)
            quant_mb = os.path.getsize(quant) / (1024 * 1024)
            ratio = quant_mb / orig_mb * 100
            print(f"{name}: {orig_mb:.1f} MB -> {quant_mb:.1f} MB ({ratio:.0f}%)")
else:
    print("Skipping quantized conversion - checkpoint not available.")

## 9. TFLite 모델로 추론 (Inference)

변환된 TFLite 모델을 사용하여 1024×1024 이미지를 생성합니다.

### SDXL 추론 파이프라인:
1. **Tokenize** - 프롬프트를 토큰으로 변환
2. **Dual Encode** - CLIP-L + OpenCLIP-G 인코딩, context 결합
3. **Add Embedding** - pooled text + time_ids 임베딩 생성
4. **Diffusion Loop** - N 스텝 노이즈 제거 (context + add_emb 사용)
5. **VAE Decode** - 잠재 텐서 → 이미지

In [ ]:
from litert_torch.generative.examples.stable_diffusion import samplers
from litert_torch.generative.examples.stable_diffusion import tokenizer
import time


class SDXLTFLitePipeline:
    """TFLite 모델 기반 SDXL 파이프라인."""

    def __init__(
        self,
        tokenizer_vocab_dir: str,
        clip_tflite_path: str,
        open_clip_tflite_path: str,
        diffusion_tflite_path: str,
        decoder_tflite_path: str,
    ):
        self.tokenizer = tokenizer.Tokenizer(tokenizer_vocab_dir)
        self.clip = litert_torch.load(clip_tflite_path)
        self.open_clip = litert_torch.load(open_clip_tflite_path)
        self.diffusion = litert_torch.load(diffusion_tflite_path)
        self.decoder = litert_torch.load(decoder_tflite_path)
        print("All TFLite models loaded.")


def generate_sdxl_image(
    model,
    prompt: str,
    output_path: str,
    uncond_prompt: str = "",
    cfg_scale: float = 7.5,
    height: int = 1024,
    width: int = 1024,
    sampler_name: str = "k_euler",
    n_inference_steps: int = 20,
    seed: int = None,
):
    """TFLite 모델로 SDXL 이미지를 생성합니다."""
    if seed is not None:
        np.random.seed(seed)

    # Sampler
    if sampler_name == "k_euler":
        sampler = samplers.KEulerSampler(n_inference_steps=n_inference_steps)
    elif sampler_name == "k_euler_ancestral":
        sampler = samplers.KEulerAncestralSampler(n_inference_steps=n_inference_steps)
    elif sampler_name == "k_lms":
        sampler = samplers.KLMSSampler(n_inference_steps=n_inference_steps)
    else:
        raise ValueError(f"Unknown sampler: {sampler_name}")

    # Step 1: Dual text encoding
    print("Step 1: Dual text encoding (CLIP-L + OpenCLIP-G)...")
    cond_tokens = np.array(model.tokenizer.encode(prompt)).astype(np.int32)
    uncond_tokens = np.array(model.tokenizer.encode(uncond_prompt)).astype(np.int32)

    cond_clip = model.clip(cond_tokens, signature_name="encode")
    uncond_clip = model.clip(uncond_tokens, signature_name="encode")

    cond_open_hidden, cond_pooled = model.open_clip(cond_tokens, signature_name="encode")
    uncond_open_hidden, uncond_pooled = model.open_clip(uncond_tokens, signature_name="encode")

    cond_context = np.concatenate([cond_clip, cond_open_hidden], axis=-1)
    uncond_context = np.concatenate([uncond_clip, uncond_open_hidden], axis=-1)
    context = np.concatenate([cond_context, uncond_context], axis=0)

    # Step 2: Add embedding
    print("Step 2: Computing add_embedding...")
    add_time_ids = util.get_add_time_ids(
        original_size=(height, width),
        crop_coords=(0, 0),
        target_size=(height, width),
    )
    time_ids_emb = util.encode_add_time_ids(add_time_ids).numpy()
    cond_add_emb = np.concatenate([cond_pooled, time_ids_emb], axis=-1)
    uncond_add_emb = np.concatenate([uncond_pooled, time_ids_emb], axis=-1)
    add_emb = np.concatenate([cond_add_emb, uncond_add_emb], axis=0).astype(np.float32)

    # Step 3: Initialize latents
    print("Step 3: Initializing latents...")
    noise_shape = (1, 4, height // 8, width // 8)
    latents = np.random.normal(size=noise_shape).astype(np.float32)
    latents *= sampler.initial_scale

    # Step 4: Diffusion loop
    print(f"Step 4: Diffusion ({n_inference_steps} steps)...")
    timesteps = tqdm.tqdm(sampler.timesteps, desc="Denoising")
    for _, timestep in enumerate(timesteps):
        time_embedding = util.get_time_embedding(timestep).numpy()
        input_latents = latents * sampler.get_input_scale()
        input_latents = input_latents.repeat(2, axis=0)

        output = model.diffusion(
            input_latents.astype(np.float32),
            context.astype(np.float32),
            time_embedding.astype(np.float32),
            add_emb,
            signature_name="diffusion",
        )
        output_cond, output_uncond = np.split(output, 2, axis=0)
        output = cfg_scale * (output_cond - output_uncond) + output_uncond
        latents = sampler.step(latents, output)

    # Step 5: Decode
    print("Step 5: Decoding to image...")
    images = model.decoder(latents.astype(np.float32), signature_name="decode")
    images = util.rescale(images, (-1, 1), (0, 255), clamp=True)
    images = util.move_channel(images, to="last")

    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    result_image = Image.fromarray(images[0].astype(np.uint8))
    result_image.save(output_path)
    print(f"Image saved to: {output_path}")
    return result_image


print("SDXL inference pipeline defined.")

In [ ]:
# TFLite 추론 실행
tflite_files = {
    "clip": f"{OUTPUT_DIR}/clip.tflite",
    "open_clip": f"{OUTPUT_DIR}/open_clip.tflite",
    "diffusion": f"{OUTPUT_DIR}/diffusion.tflite",
    "decoder": f"{OUTPUT_DIR}/decoder.tflite",
}
all_exist = all(os.path.exists(p) for p in tflite_files.values())
tokenizer_exists = os.path.exists(TOKENIZER_DIR)

if all_exist and tokenizer_exists:
    sdxl_pipeline = SDXLTFLitePipeline(
        tokenizer_vocab_dir=TOKENIZER_DIR,
        clip_tflite_path=tflite_files["clip"],
        open_clip_tflite_path=tflite_files["open_clip"],
        diffusion_tflite_path=tflite_files["diffusion"],
        decoder_tflite_path=tflite_files["decoder"],
    )

    start_time = time.time()
    result = generate_sdxl_image(
        model=sdxl_pipeline,
        prompt="a photograph of an astronaut riding a horse on the moon, high quality, detailed",
        output_path=f"{OUTPUT_DIR}/generated_image.jpg",
        sampler_name="k_euler",
        n_inference_steps=20,
        seed=42,
    )
    print(f"Total time: {time.time() - start_time:.1f}s")
else:
    missing = []
    if not all_exist:
        missing.append("TFLite models")
    if not tokenizer_exists:
        missing.append(f"tokenizer dir ({TOKENIZER_DIR})")
    print(f"Skipping inference - missing: {', '.join(missing)}")

In [ ]:
# 생성된 이미지 표시
output_image_path = f"{OUTPUT_DIR}/generated_image.jpg"
if os.path.exists(output_image_path):
    from IPython.display import display
    display(Image.open(output_image_path))
else:
    print("No generated image found.")

## 10. Playground

다양한 프롬프트와 설정으로 SDXL 이미지를 생성합니다.

In [ ]:
# === 배치 이미지 생성 ===
PROMPTS = [
    "a beautiful sunset over a calm ocean, oil painting, vivid colors, high resolution",
    "a cute cat sitting on a windowsill, digital art, detailed fur, 4k",
    "futuristic cyberpunk cityscape at night, neon lights, rain, cinematic",
    "a cozy cabin in snowy mountains, watercolor style, peaceful",
]

NEGATIVE_PROMPT = "blurry, low quality, deformed, ugly, bad anatomy"
CFG_SCALE = 7.5
SAMPLER = "k_euler"
N_STEPS = 20
SEED = 42

generated_images = []
generation_times = []

if "sdxl_pipeline" in dir() and sdxl_pipeline is not None:
    for i, prompt in enumerate(PROMPTS):
        print(f"\n{'='*60}")
        print(f"[{i+1}/{len(PROMPTS)}] \"{prompt}\"")
        print(f"{'='*60}")

        output_path = f"{OUTPUT_DIR}/playground_{i:02d}.png"
        start = time.time()
        img = generate_sdxl_image(
            model=sdxl_pipeline,
            prompt=prompt,
            output_path=output_path,
            uncond_prompt=NEGATIVE_PROMPT,
            cfg_scale=CFG_SCALE,
            sampler_name=SAMPLER,
            n_inference_steps=N_STEPS,
            seed=SEED + i if SEED is not None else None,
        )
        elapsed = time.time() - start
        generation_times.append(elapsed)
        generated_images.append(img)
        print(f"  Time: {elapsed:.1f}s")

    print(f"\nAll {len(PROMPTS)} images generated!")
    print(f"Average time per image: {np.mean(generation_times):.1f}s")
else:
    print("Pipeline not loaded. Run the inference section first.")

In [ ]:
# 생성된 이미지를 그리드로 표시
import math

if generated_images:
    n_images = len(generated_images)
    cols = min(n_images, 2)
    rows = math.ceil(n_images / cols)
    display_size = 512

    grid_w = cols * display_size
    grid_h = rows * display_size
    grid = Image.new("RGB", (grid_w, grid_h), color=(255, 255, 255))

    for idx, img in enumerate(generated_images):
        r, c = divmod(idx, cols)
        grid.paste(img.resize((display_size, display_size)), (c * display_size, r * display_size))

    for i, prompt in enumerate(PROMPTS):
        print(f"[{i}] {prompt}")
    print()

    from IPython.display import display
    display(grid)
    grid.save(f"{OUTPUT_DIR}/playground_grid.png")
    print(f"\nGrid saved to: {OUTPUT_DIR}/playground_grid.png")
else:
    print("No images generated yet.")

### CFG Scale 비교

- **낮은 값 (1-3)**: 프롬프트를 느슨하게 따름
- **중간 값 (7-8)**: 일반적으로 최적 품질
- **높은 값 (15+)**: 프롬프트에 강하게 집착, 과포화 가능

In [ ]:
# CFG Scale 비교 실험
CFG_PROMPT = "a beautiful landscape with mountains and a lake, photorealistic, 4k"
CFG_VALUES = [2.0, 7.5, 15.0]
cfg_images = []

if "sdxl_pipeline" in dir() and sdxl_pipeline is not None:
    for cfg_val in CFG_VALUES:
        print(f"\nGenerating with CFG scale = {cfg_val}...")
        img = generate_sdxl_image(
            model=sdxl_pipeline,
            prompt=CFG_PROMPT,
            output_path=f"{OUTPUT_DIR}/cfg_{cfg_val:.1f}.png",
            uncond_prompt="blurry, low quality",
            cfg_scale=cfg_val,
            sampler_name="k_euler",
            n_inference_steps=20,
            seed=42,
        )
        cfg_images.append(img)

    display_size = 512
    total_w = display_size * len(cfg_images)
    comparison = Image.new("RGB", (total_w, display_size))
    for i, img in enumerate(cfg_images):
        comparison.paste(img.resize((display_size, display_size)), (i * display_size, 0))

    print(f"\nCFG Scale: {' | '.join(str(v) for v in CFG_VALUES)}")
    from IPython.display import display
    display(comparison)
    comparison.save(f"{OUTPUT_DIR}/cfg_comparison.png")
else:
    print("Pipeline not loaded.")

### Sampler 비교

- **k_euler**: 빠르고 안정적
- **k_euler_ancestral**: 더 다양한 결과
- **k_lms**: Linear Multistep, 부드러운 결과

In [ ]:
# Sampler 비교 실험
SAMPLER_PROMPT = "a portrait of a wise old wizard, fantasy art, detailed, epic lighting"
SAMPLERS_TO_COMPARE = ["k_euler", "k_euler_ancestral", "k_lms"]
sampler_images = []

if "sdxl_pipeline" in dir() and sdxl_pipeline is not None:
    for s_name in SAMPLERS_TO_COMPARE:
        print(f"\nGenerating with sampler = {s_name}...")
        img = generate_sdxl_image(
            model=sdxl_pipeline,
            prompt=SAMPLER_PROMPT,
            output_path=f"{OUTPUT_DIR}/sampler_{s_name}.png",
            uncond_prompt="blurry, low quality",
            cfg_scale=7.5,
            sampler_name=s_name,
            n_inference_steps=20,
            seed=42,
        )
        sampler_images.append(img)

    display_size = 512
    total_w = display_size * len(sampler_images)
    comparison = Image.new("RGB", (total_w, display_size))
    for i, img in enumerate(sampler_images):
        comparison.paste(img.resize((display_size, display_size)), (i * display_size, 0))

    print(f"\nSamplers: {' | '.join(SAMPLERS_TO_COMPARE)}")
    from IPython.display import display
    display(comparison)
    comparison.save(f"{OUTPUT_DIR}/sampler_comparison.png")
else:
    print("Pipeline not loaded.")

## 요약

이 노트북에서 다룬 내용:

1. **듀얼 텍스트 인코더** (CLIP-L + OpenCLIP-G)로 2048-dim context 생성
2. **MultiLayerTransformerBlock2D**로 SDXL UNet의 다중 레이어 트랜스포머 구현
3. **AddEmbedding**으로 pooled text + time_ids 추가 조건 처리
4. 4개 컴포넌트를 **TFLite로 변환** (clip, open_clip, diffusion, decoder)
5. `quant_recipes`를 사용한 **양자화** (weight-only INT8)
6. 1024×1024 **이미지 생성 파이프라인** 실행

### 다음 단계
- 다른 양자화 레시피 실험 (INT4, FP16 등)
- Android/iOS 앱에 TFLite 모델 통합
- SDXL Refiner 모델 추가 포팅
- Model Explorer로 변환된 모델 시각화 및 검증